# Sternberg (1966): analysis

Computes the paper's measures from this folder's run files (`*_trials.csv`):

- mean reaction time against the number of digits in memory, **s**, for positive and negative test digits (Figs. 1 and 2);
- the least-squares line through the overall means per s: **slope** (the comparison time, ms per digit) and **zero intercept**, and the share of variance it accounts for;
- separate lines for positive and negative responses and the difference in their slopes (equal slopes = exhaustive scanning; a self-terminating scan predicts the positive slope at half the negative);
- error rates (wrong key; imperfect recall in experiment 1);
- the upper bound for a parallel process with independent comparison times (broken curves in the figures), and a log s fit for comparison (paper note 17).

The paper's analysis (notes 5-7): trials with the wrong lever are excluded; each subject's mean latency per s is computed; quantities are averaged over subjects. Its standard errors came from variance components of an analysis of variance with subjects as random effects; here the standard error of a slope or intercept is the between-subject standard deviation of the per-subject estimates divided by sqrt(n) (a deviation, docs/methods.md), and it needs at least two subjects.

Point `ANALYSIS_DATA` (environment variable or the cell below) at a data root or a run folder. Every `*_trials.csv` below it is read; the subject is taken from the `sub-<id>` folder. Data from `--mode simulate` comes from a simulated subject whose reaction times were generated from the paper's own fitted line: recovering the paper's numbers from it checks this pipeline, it is not a replication.

In [ ]:
import os
from pathlib import Path

import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

HERE = Path.cwd() if (Path.cwd() / 'run.py').exists() else Path.cwd().parent
DATA = os.environ.get('ANALYSIS_DATA')
if DATA is None:
    roots = [HERE / 'data', HERE / 'data-rehearsal']
    DATA = next((r for r in roots if r.is_dir() and any(r.rglob('*_trials.csv'))), None)
FIGURES = Path(os.environ.get('ANALYSIS_FIGURES', HERE / 'notebooks' / 'figures'))
# The paper excluded wrong-lever trials (note 5). It does not say whether trials with an
# imperfect recall were excluded too: ASSUMED kept (set True to drop them).
EXCLUDE_RECALL_FAILURES = False
print('data:', DATA)

In [ ]:
def load(root):
    frames = []
    for path in sorted(Path(root).rglob('*_trials.csv')):
        df = pd.read_csv(path)
        subject = next((p[4:] for p in path.parts if p.startswith('sub-')), path.parent.name)
        df['subject'] = subject
        df['run'] = path.parent.name
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

trials = load(DATA) if DATA else pd.DataFrame()
if len(trials):
    print(trials.groupby(['procedure', 'subject', 'stage']).size().rename('trials').to_string())
else:
    print('no trials found')

In [ ]:
PAPER = {  # what the paper reports (p. 652-653), for comparison only
    'varied-set': {'slope': 37.9, 'intercept': 397.2, 'error_pct': 1.3, 'recall_imperfect_pct': 1.4},
    'fixed-set': {'slope': 38.3, 'intercept': 369.4, 'error_pct': 1.0},
}

def fit_line(s, rt):
    s, rt = np.asarray(s, float), np.asarray(rt, float)
    if len(np.unique(s)) < 2:
        return np.nan, np.nan, np.nan
    slope, intercept = np.polyfit(s, rt, 1)
    pred = intercept + slope * s
    ss_tot = ((rt - rt.mean()) ** 2).sum()
    r2 = 1 - ((rt - pred) ** 2).sum() / ss_tot if ss_tot > 0 else np.nan
    return slope, intercept, r2

def analysis_trials(df):
    # Test trials that produced a response (a timed-out trial was served again).
    t = df[(df['stage'] == 'test') & df['completed'].astype(bool)].copy()
    t['correct'] = t['correct'].astype(str).str.lower().eq('true')
    return t

def latency_table(t):
    # The paper's unit: each subject's mean latency per s (and per response type), correct
    # responses only; then the mean over subjects.
    ok = t[t['correct']]
    if EXCLUDE_RECALL_FAILURES and 'recall_correct' in ok:
        ok = ok[ok['recall_correct'].astype(str).str.lower().eq('true')]
    by_type = ok.groupby(['subject', 'set_size', 'probe_type'])['rt_ms'].mean().unstack('probe_type')
    overall = ok.groupby(['subject', 'set_size'])['rt_ms'].mean().rename('overall')
    per_subject = by_type.join(overall)
    return per_subject, per_subject.groupby('set_size').mean(), ok

In [ ]:
results = {}
for procedure, df in (trials.groupby('procedure') if len(trials) else []):
    t = analysis_trials(df)
    per_subject, means, ok = latency_table(t)
    s = means.index.to_numpy()
    res = {'subjects': t['subject'].nunique(), 'test_trials': len(t),
           'error_pct': 100 * (~t['correct']).mean() if len(t) else np.nan}
    if 'recall_correct' in t:
        res['recall_imperfect_pct'] = 100 * (~t['recall_correct'].astype(str).str.lower().eq('true')).mean()
    res['slope'], res['intercept'], res['r2'] = fit_line(s, means['overall'])
    for kind in ('positive', 'negative'):
        if kind in means:
            m = means[kind].dropna()
            res[f'slope_{kind}'], res[f'intercept_{kind}'], _ = fit_line(m.index, m.values)
    res['slope_difference'] = res.get('slope_positive', np.nan) - res.get('slope_negative', np.nan)
    if procedure == 'varied-set' and {'positive', 'negative'} <= set(means):
        if 1 in means.index:
            res['neg_minus_pos_at_s1'] = means.loc[1, 'negative'] - means.loc[1, 'positive']
        rest = means[means.index >= 2]
        sp, _, _ = fit_line(rest.index, rest['positive'])
        sn, _, _ = fit_line(rest.index, rest['negative'])
        res['slope_difference_s2plus'] = sp - sn
    # Per-subject lines, and their spread across subjects (needs two or more subjects).
    subj = []
    for sid, g in per_subject.groupby(level='subject'):
        b, a, _ = fit_line(g.index.get_level_values('set_size'), g['overall'])
        subj.append({'subject': sid, 'slope': b, 'intercept': a})
    subj = pd.DataFrame(subj)
    n = subj['slope'].notna().sum()
    res['slope_se'] = subj['slope'].std(ddof=1) / np.sqrt(n) if n >= 2 else np.nan
    res['intercept_se'] = subj['intercept'].std(ddof=1) / np.sqrt(n) if n >= 2 else np.nan
    # Log s fit (note 17): the parallel models with exponential comparison times predict a
    # function roughly linear in log s.
    _, _, res['r2_log_s'] = fit_line(np.log(s), means['overall']) if len(s) > 1 else (np.nan,) * 3
    # Upper bound for a parallel process with independent, identically distributed comparison
    # times (Hartley & David, 1954; paper notes 15-16), applied to negative responses: with
    # the mean m1 and standard deviation sd1 of the s = 1 latencies, E[max of s] <= m1 + sd1 (s-1)/sqrt(2s-1).
    # Our reading of the bound; Sternberg's own application was "in preparation".
    neg1 = ok[(ok['set_size'] == 1) & (ok['probe_type'] == 'negative')]['rt_ms']
    if len(neg1) >= 2:
        res['parallel_bound'] = {int(k): neg1.mean() + neg1.std(ddof=1) * (k - 1) / np.sqrt(2 * k - 1)
                                 for k in s}
    results[procedure] = (res, means, per_subject, subj)
    print(f'== {procedure}')
    print(means.round(1).to_string())
    show = {k: (round(v, 2) if isinstance(v, float) else v) for k, v in res.items() if k != 'parallel_bound'}
    for k, v in show.items():
        print(f'  {k}: {v}')

In [ ]:
rows = []
for procedure, (res, *_rest) in results.items():
    for key, paper in PAPER[procedure].items():
        rows.append({'procedure': procedure, 'measure': key, 'this data': round(res.get(key, np.nan), 2),
                     'paper': paper})
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False) if len(comparison) else 'nothing to compare')

In [ ]:
FIGURES.mkdir(parents=True, exist_ok=True)
for procedure, (res, means, per_subject, subj) in results.items():
    fig, ax = plt.subplots(figsize=(5, 4))
    s = means.index.to_numpy(float)
    if 'positive' in means:
        ax.plot(s, means['positive'], 'o', color='k', label='positive (filled)')
    if 'negative' in means:
        ax.plot(s, means['negative'], 'o', mfc='white', color='k', label='negative (open)')
    ax.hlines(means['overall'], s - 0.15, s + 0.15, color='k', lw=3, label='overall mean')
    if np.isfinite(res['slope']):
        xx = np.linspace(s.min(), s.max(), 50)
        ax.plot(xx, res['intercept'] + res['slope'] * xx, 'k-', lw=1,
                label=f"fit: {res['intercept']:.0f} + {res['slope']:.1f} s")
    paper = PAPER[procedure]
    ax.plot(s, paper['intercept'] + paper['slope'] * s, ':', color='0.5',
            label=f"paper: {paper['intercept']:.0f} + {paper['slope']:.1f} s")
    if 'parallel_bound' in res:
        pb = res['parallel_bound']
        ax.plot(list(pb), list(pb.values()), '--', color='0.3', label='parallel bound (neg.)')
    ax.set_xlabel('number of digits in memory, s' if procedure == 'varied-set' else 'size of positive set, s')
    ax.set_ylabel('mean reaction time (ms)')
    ax.set_title(f'{procedure}: {res["subjects"]} subject(s)')
    ax.legend(fontsize=7, frameon=False)
    fig.tight_layout()
    fig.savefig(FIGURES / f'{procedure}.png', dpi=120)
    plt.close(fig)
    print('figure:', FIGURES / f'{procedure}.png')

Reading the result: the slope is the mean comparison time per digit (the paper: about 38 ms, a rate of 25-30 digits per second); the intercept is everything that does not depend on s. Equal positive and negative slopes indicate exhaustive scanning. With fewer than two subjects there is no between-subject standard error.